# Phase: Label + Feature Engineering (Next 3 Hours, Multi-class)

**Goal:** Build a supervised HOOD × 3-hour dataset for forecasting the **next 3-hour block** risk class.

**Inputs (in `processed_data/`):**
- `model_hood_3h_weather.csv`  (HOOD×3h merged, may contain only non-zero collision blocks)
- `weather_3h.csv` (complete 3-hour weather timeline)

**Outputs (saved to `processed_data/`):**
- `supervised_hood_3h_multiclass.csv`

> Tip: This notebook rebuilds the full HOOD×time grid (including zero-collision blocks). That can be large (~1.3M rows).


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"

MODEL_PATH = DATA_DIR / "model_hood_3h_weather.csv"
WEATHER3H_PATH = DATA_DIR / "weather_3h.csv"

OUT_SUPERVISED = DATA_DIR / "supervised_hood_3h_multiclass.csv"

# Parameters (safe defaults)
FREQ = "3h"                 # 3-hour blocks
HIGH_RISK_THRESHOLD = 2     # class 2 means >=2 collisions
LAGS = [1, 2, 8]            # 3h, 6h, 24h
ROLL_WINDOWS = [4, 8]       # 12h, 24h

In [6]:
# --- Load inputs ---
base = pd.read_csv(MODEL_PATH, low_memory=False)
base["time_3h"] = pd.to_datetime(base["time_3h"], errors="coerce")
base["HOOD_158_CODE"] = base["HOOD_158_CODE"].astype(str).str.zfill(3)

weather_3h = pd.read_csv(WEATHER3H_PATH, low_memory=False)
weather_3h["time_3h"] = pd.to_datetime(weather_3h["time_3h"], errors="coerce")
weather_3h = weather_3h.sort_values("time_3h").drop_duplicates("time_3h", keep="first")

print("Base shape:", base.shape)
print("Weather_3h shape:", weather_3h.shape)
print("HOODs in base:", base["HOOD_158_CODE"].nunique())
print("Time range:", weather_3h["time_3h"].min(), "to", weather_3h["time_3h"].max())

Base shape: (148208, 21)
Weather_3h shape: (8768, 10)
HOODs in base: 158
Time range: 2023-01-01 00:00:00 to 2025-12-31 21:00:00


In [7]:
# --- Build full HOOD × time grid (adds zero-collision blocks) ---
hoods = sorted(base["HOOD_158_CODE"].unique())
t_min, t_max = weather_3h["time_3h"].min(), weather_3h["time_3h"].max()
times = pd.date_range(t_min, t_max, freq=FREQ)

grid = pd.MultiIndex.from_product([hoods, times], names=["HOOD_158_CODE", "time_3h"]).to_frame(index=False)

# Keep collision count columns from base (if present)
collision_cols = [
    "collisions", "injury_collisions", "ftr_collisions", "pd_collisions",
    "pedestrian_collisions", "bicycle_collisions"
]
collision_cols = [c for c in collision_cols if c in base.columns]
base_small = base[["HOOD_158_CODE", "time_3h"] + collision_cols].copy()

df = grid.merge(base_small, on=["HOOD_158_CODE", "time_3h"], how="left")

# Fill missing blocks with zeros
for c in collision_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int32")

# Merge weather (same weather for all HOODs at same time)
df = df.merge(weather_3h, on="time_3h", how="left", validate="m:1")

# Fill any weather missing at edges (rare)
wx_cols = [c for c in weather_3h.columns if c != "time_3h"]
for c in wx_cols:
    if c in df.columns and df[c].dtype.kind in "fc":
        df[c] = df[c].interpolate(limit_direction="both")

df = df.sort_values(["HOOD_158_CODE", "time_3h"]).reset_index(drop=True)

print("Full grid shape:", df.shape)
print("Collision min/max:", int(df["collisions"].min()), int(df["collisions"].max()))
print("Zero share:", round(float((df["collisions"]==0).mean()), 4))


Full grid shape: (1385344, 17)
Collision min/max: 0 17
Zero share: 0.893


In [8]:
# --- Time features (numeric + cyclical) ---
df["block_hour"] = df["time_3h"].dt.hour
df["dow_num"] = df["time_3h"].dt.dayofweek
df["month_num"] = df["time_3h"].dt.month
df["is_weekend"] = (df["dow_num"] >= 5).astype("uint8")

# Cyclical encodings (often improves models)
df["hour_sin"] = np.sin(2*np.pi*df["block_hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["block_hour"]/24)

df["dow_sin"] = np.sin(2*np.pi*df["dow_num"]/7)
df["dow_cos"] = np.cos(2*np.pi*df["dow_num"]/7)

df["month_sin"] = np.sin(2*np.pi*df["month_num"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month_num"]/12)


In [9]:
# --- Lag + rolling features (NO leakage) ---
g = df.groupby("HOOD_158_CODE", sort=False)

# Collision lags
for lag in LAGS:
    df[f"coll_lag_{lag}"] = g["collisions"].shift(lag)

# Type/user lags (optional)
for col in ["injury_collisions", "ftr_collisions", "pd_collisions", "pedestrian_collisions", "bicycle_collisions"]:
    if col in df.columns:
        df[f"{col}_lag_1"] = g[col].shift(1)

# Rolling mean/sum on shifted series (past only)
s = g["collisions"].shift(1)
for w in ROLL_WINDOWS:
    df[f"coll_roll_mean_{w}"] = s.groupby(df["HOOD_158_CODE"]).rolling(w, min_periods=w).mean().reset_index(level=0, drop=True)
    df[f"coll_roll_sum_{w}"]  = s.groupby(df["HOOD_158_CODE"]).rolling(w, min_periods=w).sum().reset_index(level=0, drop=True)

# Optional: weather lag (delayed effects)
for wx in ["temperature","visibility","rain","snow","snow_on_ground","wind_speed","relative_humidity","pressure_sea","cloud_cover_8"]:
    if wx in df.columns:
        df[f"{wx}_lag_1"] = g[wx].shift(1)


In [10]:
# --- Create multi-class label for NEXT 3-hour block ---
df["y_count_next"] = g["collisions"].shift(-1)

# y_class: 0 / 1 / 2(>=2)
df["y_class"] = np.select(
    [
        df["y_count_next"].isna(),
        df["y_count_next"] == 0,
        df["y_count_next"] == 1,
        df["y_count_next"] >= HIGH_RISK_THRESHOLD,
    ],
    [np.nan, 0, 1, 2],
    default=np.nan
)


In [11]:
# --- Final filtering: drop rows without label and without required history ---
# Drop end-of-series rows where y is missing
df = df[df["y_class"].notna()].copy()

# Require core lag/rolling features
required = ["coll_lag_1", "coll_lag_2", "coll_lag_8", "coll_roll_mean_4", "coll_roll_mean_8"]
required = [c for c in required if c in df.columns]
df = df.dropna(subset=required).copy()

# Cast label to integer
df["y_class"] = df["y_class"].astype("int8")

# Optional: HOOD categorical to reduce memory
df["HOOD_158_CODE"] = df["HOOD_158_CODE"].astype("category")

print("Supervised shape:", df.shape)
print("Class distribution:")
print(df["y_class"].value_counts(normalize=True).sort_index().round(4))


Supervised shape: (1383922, 50)
Class distribution:
y_class
0    0.8930
1    0.0927
2    0.0143
Name: proportion, dtype: float64


In [12]:
# --- Save supervised dataset ---
df.to_csv(OUT_SUPERVISED, index=False, float_format="%.3f")
print("Saved:", OUT_SUPERVISED)


Saved: ..\data\processed\supervised_hood_3h_multiclass.csv
